In [ ]:
# Import requirements
import cv2
import json
import numpy as np
import mediapipe as mp
import tensorflow as tf
from collections import deque, Counter
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import time

LANDMARKER_MODEL_PATH = r"D:\lakehead\Deep Learning\project\full_house\hand_landmarker.task"
CLASSIFIER_MODEL_PATH = "mediapipe_sad_model_handedness_fixed.keras"
LABELS_PATH = "mediapipe_labels_handedness_fixed.json"

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    class_names = json.load(f)

classifier = tf.keras.models.load_model(CLASSIFIER_MODEL_PATH)

#apply the same options for the detection same as when we got the landmarks
base_options = python.BaseOptions(model_asset_path=LANDMARKER_MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

# same normalization for the model
def normalize_landmarks(sample, handed_label="Unknown"):
    sample = np.array(sample, dtype=np.float32).reshape(21, 3)

    if handed_label == "Right":
        sample[:, 0] = 1.0 - sample[:, 0]

    wrist = sample[0].copy()
    sample = sample - wrist

    max_val = np.max(np.abs(sample))
    if max_val > 0:
        sample = sample / max_val

    return sample.flatten()

cap = cv2.VideoCapture(0)
prediction_buffer = deque(maxlen=7)

typed_text = ""
last_added_letter = ""

# slower writing settings
min_confidence_to_write = 0.90
# hold the same sign for 1.5 seconds before writing
hold_time_required = 1.5   
blank_reset_required = True

current_candidate = None
candidate_start_time = None
ready_for_next_letter = True

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        display = frame.copy()

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
        result = landmarker.detect(mp_image)

        smooth_label = "Uncertain"
        confidence = 0.0
        handed_label = "Unknown"
        probs_text = ""
        progress = 0.0

        if result.hand_landmarks and len(result.hand_landmarks) > 0:
            hand_landmarks = result.hand_landmarks[0]

            if result.handedness and len(result.handedness) > 0:
                handed_label = result.handedness[0][0].category_name

            h, w, _ = frame.shape
            xs = [lm.x for lm in hand_landmarks]
            ys = [lm.y for lm in hand_landmarks]

            x_min = max(0, int(min(xs) * w) - 20)
            y_min = max(0, int(min(ys) * h) - 20)
            x_max = min(w, int(max(xs) * w) + 20)
            y_max = min(h, int(max(ys) * h) + 20)

            cv2.rectangle(display, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

            row = []
            for lm in hand_landmarks:
                row.extend([lm.x, lm.y, lm.z])

            X_input = normalize_landmarks(row, handed_label).reshape(1, -1)
            
            # get the prediction based on the detected hand
            preds = classifier.predict(X_input, verbose=0)[0]
            pred_idx = int(np.argmax(preds))
            confidence = float(preds[pred_idx])
            pred_label = class_names[pred_idx]

            prediction_buffer.append(pred_label)
            smooth_label = Counter(prediction_buffer).most_common(1)[0][0]

            if confidence < 0.80:
                smooth_label = "Uncertain"

            probs_text = " | ".join(
                [f"{class_names[i]}: {preds[i]:.2f}" for i in range(len(class_names))]
            )

            # slower detection / writing logic
            now = time.time()

            if smooth_label != "Uncertain" and confidence >= min_confidence_to_write:
                if ready_for_next_letter:
                    if current_candidate != smooth_label:
                        current_candidate = smooth_label
                        candidate_start_time = now
                    else:
                        elapsed = now - candidate_start_time
                        progress = min(elapsed / hold_time_required, 1.0)

                        if elapsed >= hold_time_required:
                            typed_text += smooth_label
                            last_added_letter = smooth_label
                            ready_for_next_letter = False
                            current_candidate = None
                            candidate_start_time = None
                else:
                    # waiting for hand removal or uncertain state before allowing next letter
                    progress = 1.0
            else:
                current_candidate = None
                candidate_start_time = None

            cv2.putText(
                display,
                f"Prediction: {smooth_label} ({confidence*100:.1f}%)",
                (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                display,
                f"Handedness: {handed_label}",
                (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 0),
                2
            )

            cv2.putText(
                display,
                probs_text,
                (10, 95),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2
            )
        else:
            cv2.putText(
                display,
                "No hand detected",
                (10, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            # reset so the next letter can be added
            if blank_reset_required:
                ready_for_next_letter = True
                current_candidate = None
                candidate_start_time = None
                progress = 0.0

        # progress bar
        bar_x1, bar_y1 = 10, 110
        bar_x2, bar_y2 = 310, 135
        cv2.rectangle(display, (bar_x1, bar_y1), (bar_x2, bar_y2), (255, 255, 255), 2)
        fill_width = int((bar_x2 - bar_x1) * progress)
        cv2.rectangle(display, (bar_x1, bar_y1), (bar_x1 + fill_width, bar_y2), (0, 255, 0), -1)
        cv2.putText(
            display,
            "Hold sign to type",
            (320, 128),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2
        )

        # text area
        cv2.rectangle(display, (10, 150), (1250, 210), (0, 0, 0), -1)
        cv2.putText(
            display,
            f"Text: {typed_text}",
            (20, 190),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2
        )

        cv2.putText(
            display,
            "Q=Quit | SPACE=Add space | C=Clear | B=Backspace ",
            (10, 240),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2
        )

        cv2.imshow("MediaPipe A/D/S - Left and Right Hand", display)

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break
        elif key == 32:  # space
            typed_text += " "
            ready_for_next_letter = True
            current_candidate = None
            candidate_start_time = None
        elif key == ord("c"):
            typed_text = ""
            ready_for_next_letter = True
            current_candidate = None
            candidate_start_time = None
        elif key == ord("b"):
            typed_text = typed_text[:-1]
            ready_for_next_letter = True
            current_candidate = None
            candidate_start_time = None

cap.release()
cv2.destroyAllWindows()